In [1]:
!rm -rf InkubaLM-Challenge
!git clone https://github.com/melissafasol/InkubaLM-Challenge.git
%cd InkubaLM-Challenge

Cloning into 'InkubaLM-Challenge'...
remote: Enumerating objects: 242, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 242 (delta 17), reused 18 (delta 5), pack-reused 202 (from 1)
Receiving objects: 100% (242/242), 979.18 KiB | 13.06 MiB/s, done.
Resolving deltas: 100% (144/144), done.
/content/InkubaLM-Challenge


In [114]:
!git checkout refactor-src-structure
!git pull

Already on 'refactor-src-structure'
Your branch is up to date with 'origin/refactor-src-structure'.
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 541 bytes | 270.00 KiB/s, done.
From https://github.com/melissafasol/InkubaLM-Challenge
   85bb456..8dd1189  refactor-src-structure -> origin/refactor-src-structure
Updating 85bb456..8dd1189
Fast-forward
 src/data_augment.py |  70 ++-------------------------
 src/model_utils.py  | 188 ++----------------------------------------------------------------------
 2 files changed, 8 insertions(+), 250 deletions(-)


In [3]:
!pip install datasets
!pip install transformers datasets peft trl accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "

In [115]:
import sys
sys.path.append("..")  # Add parent directory to the path

import os
from typing import List
from pathlib import Path
import numpy as np

# DO NOT EDIT
# create submission file
import pandas as pd
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
)

from utils import (
    model_function,
    eval
    )

from src import (
    data_utils,
    model_utils,
    inference,
    prompts,
    evaluation,
    data_augment
    )

import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset, concatenate_datasets, Dataset, Value, DatasetDict

from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
from peft import PeftModel, PeftConfig
from sklearn.model_selection import train_test_split


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
#os.environ["TOKENIZERS_PARALLELISM"] = "false"

from huggingface_hub import login

try:
    from google.colab import userdata

    # Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
    # vars as appropriate for your system.
    # userdata.get("HF_TOKEN") indicates that the name of the token in the Colab env is HF_TOKEN
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except:
    os.environ["HF_TOKEN"] = "----"

login(token=os.environ["HF_TOKEN"])

token = os.environ["HF_TOKEN"]
if token == "----":
    print("⚠️ Warning: No Hugging Face token found. Some models may not load.")
else:
    login(token=token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Load Base Model and Run Inference on Test Set for Baseline Performance

In [8]:
model_name = "lelapa/InkubaLM-0.4B"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=os.environ["HF_TOKEN"])
model = model_function.load_model(model_name)

tokenizer_config.json:   0%|          | 0.00/960 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/991k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.95M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/763 [00:00<?, ?B/s]

vulavulaslm.py:   0%|          | 0.00/42.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/lelapa/InkubaLM-0.4B:
- vulavulaslm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin:   0%|          | 0.00/2.66G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [16]:
BASE_PROMPT = "Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n ### Instruction: {}\n\n ### Response: "
sent_instruction = "Please identify the sentiment reflected in this text based on the following guidelines: Positive: if a text implies positive sentiment, attitude, and emotional state. Negative: if a text implies negative sentiment or emotion. Neutral: if a text does not imply positive or negative language directly or indirectly. Provide sentiment labels only"

In [30]:
print("# Loading datasets")
sentiment_train_df = pd.read_parquet("hf://datasets/lelapa/SentimentTrain/data/train-00000-of-00001.parquet")

hau_dataset = Dataset.from_pandas(
    sentiment_train_df[sentiment_train_df['langs']=='hausa']
)
swa_dataset = Dataset.from_pandas(
    sentiment_train_df[sentiment_train_df['langs']=='swahili']
)

# If you need a DatasetDict to mimic the Hugging Face structure
dataset_dict = DatasetDict({
    "swahili": swa_dataset,
    "hausa": hau_dataset
})


# for swahili
model_function.main(
    model,
    tokenizer,
    BASE_PROMPT,
    sample_size=3,
    max_new_tokens=15,
    task_instruction=sent_instruction,
    dataset=swa_dataset,
    csv_file_path=os.path.join(
        output_path,
        "swa_sent_prediction_dev.csv"
    ),
    custom_instruct=False,
)



# Loading datasets


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [31]:
# for hausa
model_function.main(
    model,
    tokenizer,
    BASE_PROMPT,
    sample_size=3,
    task_instruction=sent_instruction,
    dataset=hau_dataset,
    csv_file_path=os.path.join(
        output_path,
        "hau_sent_prediction_dev.csv"
    ),
    max_new_tokens=15,
    custom_instruct=False,
)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [ ]:
from sklearn.utils import shuffle
test_dev = pd.concat([mt_train_og, hau_dataset_mt], axis=0, ignore_index=True)
test_dev = shuffle(test_dev, random_state=42).reset_index(drop=True)

In [12]:
hau_dataset_mt

,ID,task,langs,data_source,instruction,inputs,targets
0,opus_hausa_81,mmt,eng-hau,opus100,translate the following from English into hausa.,and we have created for them of the like there...,"kuma muka halitta musu, daga irinsa, abin da s..."
1,opus_hausa_218,mmt,eng-hau,opus100,translate the following from English into hausa.,"""come back to your lord, well-pleased (yoursel...","ka koma zuwa ga ubangijinka, alhãli kana mai y..."
2,opus_hausa_55,mmt,eng-hau,opus100,translate the following from English into hausa.,they said: thou art but one of the bewitched;,"suka ce: ""kai daga mãsu sihiri kurum kake."""
3,opus_hausa_598,mmt,eng-hau,opus100,translate the following from English into hausa.,welcome to the gnome desktop,barka da zuwa kwamfyutan tebur na gnome
4,opus_hausa_264,mmt,eng-hau,opus100,translate the following from English into hausa.,the hour of resurrection is coming. i have wil...,"""lalle ne sa'a mai zuwa ce, lnã kusa da in ɓõy..."
...,...,...,...,...,...,...,...
605,opus_hausa_71,mmt,eng-hau,opus100,translate the following from English into hausa.,as such we shall deal with the evildoers.,"lalle mũ, kamar haka muke aikatãwa game, da mã..."
606,opus_hausa_106,mmt,eng-hau,opus100,translate the following from English into hausa.,"go, you and your brother, with my signs, and d...","""ka tafi kai da ɗan'uwanka game da ãyõyina, ku..."
607,opus_hausa_270,mmt,eng-hau,opus100,translate the following from English into hausa.,when their brother lut said to them: will you ...,"a lõkacin da ɗan'uwansu, lũɗu ya ce musu, ""bã ..."
608,opus_hausa_435,mmt,eng-hau,opus100,translate the following from English into hausa.,"then they (the worshippers of idols) came, tow...","sai suka fuskanto zuwa gare shi, sunã gaggãwa."


## MT baseline

In [116]:
import os
output_path = "/content/drive/MyDrive/InkubaLM-Challenge/Output/baseline_performance/datasets"
os.makedirs(output_path, exist_ok=True)

print(os.path.exists(output_path))  # Should return True
print(output_path)

True
/content/drive/MyDrive/InkubaLM-Challenge/Output/baseline_performance/datasets


In [10]:
hau_dataset_mt = data_augment.build_hausa_mt_dataset()

📥 Loading OPUS dataset...


README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/244k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/12.2M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/246k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/97983 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [50]:
import pandas as pd
import unicodedata
import re
import string
from datasets import load_dataset
from sklearn.utils import shuffle

# -----------------------------------------------
# TEXT CLEANING UTILS
# -----------------------------------------------
def normalize(text):
    return unicodedata.normalize("NFKC", text).lower().strip()


def is_too_punctuated(text, threshold=0.3):
    punct_ratio = sum(1 for c in text if c in string.punctuation) / max(len(text), 1)
    return punct_ratio <= threshold

def is_not_religious(text):
    filter_keywords = ["o prophet", "nay,", "verily", "lo!", "messenger", "punishment", "behold", "god", "allah"]
    return not any(kw in text for kw in filter_keywords)

def is_clean(text):
    return bool(re.search(r"[a-zA-Z]", text)) and not bool(re.search(r"[�💖💁🏾👍🏾🤖📦🌍]", text))


def preprocess_dataframe(df):
    df = df.copy()

    df["inputs"] = df["inputs"].astype(str).apply(normalize)
    df["targets"] = df["targets"].astype(str).apply(normalize)

    df = df[
        df["inputs"].notna() & df["targets"].notna() &
        (df["inputs"].str.strip().str.len() > 0) &
        (df["targets"].str.strip().str.len() > 0)
    ]

    df = df[
        df["inputs"].apply(is_clean) &
        df["targets"].apply(is_clean) &
        df["inputs"].apply(is_not_religious) &
        df["targets"].str.split().str.len().ge(4) &
        (df["inputs"] != df["targets"]) &
        df["inputs"].apply(is_too_punctuated)
    ]

    df = df.drop_duplicates(subset=["inputs", "targets"])
    df = df[
        (df["inputs"].str.split().str.len() < 128) &
        (df["targets"].str.split().str.len() < 128)
    ]

    return df.reset_index(drop=True)

In [82]:
import random

# Sample English sentences for everyday conversation (can be modified later)
base_sentences = [
    "Good morning! How did you sleep?",
    "I'm going to the market. Do you want anything?",
    "What time is the meeting today?",
    "Can you help me with this?",
    "The weather is very hot today.",
    "I need to call my mother.",
    "Have you seen my keys?",
    "We should leave early to avoid traffic.",
    "I'm feeling a bit tired.",
    "Do you want to have lunch together?",
    "I like listening to music while I work.",
    "Can you pick up the kids from school?",
    "Don't forget to lock the door.",
    "Let's meet at the cafe around 4 PM.",
    "I'm learning how to cook rice perfectly.",
    "He always forgets his wallet.",
    "Would you like some tea or coffee?",
    "I'm watching a really good movie.",
    "This song reminds me of home.",
    "Can I borrow your pen?",
]

# Hausa translations (GPT-style, manually curated)
base_translations = [
    "Ina kwana! Yaya ka kwana?",
    "Zan je kasuwa. Kana bukatar wani abu?",
    "Karfe nawa taron yake yau?",
    "Za ka iya taimaka min da wannan?",
    "Yau rana tana da zafi sosai.",
    "Ina bukatar kiran mahaifiyata.",
    "Ka ga makullina?",
    "Ya kamata mu tafi da wuri domin gujewa cunkoso.",
    "Ina jin gajiya kadan.",
    "Kana son mu ci abincin rana tare?",
    "Ina jin dadin sauraron kiɗa yayin aiki.",
    "Za ka iya dauko yara daga makaranta?",
    "Kada ka manta rufe kofa.",
    "Mu hadu a gidan shayi misalin karfe 4 na yamma.",
    "Ina koyon yadda ake dafa shinkafa sosai.",
    "Yana yawan mantawa da walat dinsa.",
    "Kana son shayi ko kofi?",
    "Ina kallon fim mai kyau sosai.",
    "Wannan waƙar tana tuna min da gida.",
    "Zan iya amfani da alƙalanka?",
]

# Bootstrap 200 examples using these as base
num_samples = 200
synthetic_data = []

for i in range(num_samples):
    idx = random.randint(0, len(base_sentences) - 1)
    synthetic_data.append({
        "ID": f"bootstrap_hausa_convo_{i}",
        "task": "mmt",
        "langs": "eng-hau",
        "data_source": "synthetic",
        "instruction": "translate the following from English into Hausa.",
        "inputs": base_sentences[idx],
        "targets": base_translations[idx],
    })

df_synthetic_hausa = pd.DataFrame(synthetic_data)

In [83]:
df_synthetic_hausa

,ID,task,langs,data_source,instruction,inputs,targets
0,bootstrap_hausa_convo_0,mmt,eng-hau,synthetic,translate the following from English into Hausa.,Can you help me with this?,Za ka iya taimaka min da wannan?
1,bootstrap_hausa_convo_1,mmt,eng-hau,synthetic,translate the following from English into Hausa.,The weather is very hot today.,Yau rana tana da zafi sosai.
2,bootstrap_hausa_convo_2,mmt,eng-hau,synthetic,translate the following from English into Hausa.,The weather is very hot today.,Yau rana tana da zafi sosai.
3,bootstrap_hausa_convo_3,mmt,eng-hau,synthetic,translate the following from English into Hausa.,I'm learning how to cook rice perfectly.,Ina koyon yadda ake dafa shinkafa sosai.
4,bootstrap_hausa_convo_4,mmt,eng-hau,synthetic,translate the following from English into Hausa.,Don't forget to lock the door.,Kada ka manta rufe kofa.
...,...,...,...,...,...,...,...
195,bootstrap_hausa_convo_195,mmt,eng-hau,synthetic,translate the following from English into Hausa.,What time is the meeting today?,Karfe nawa taron yake yau?
196,bootstrap_hausa_convo_196,mmt,eng-hau,synthetic,translate the following from English into Hausa.,Good morning! How did you sleep?,Ina kwana! Yaya ka kwana?
197,bootstrap_hausa_convo_197,mmt,eng-hau,synthetic,translate the following from English into Hausa.,Can you help me with this?,Za ka iya taimaka min da wannan?
198,bootstrap_hausa_convo_198,mmt,eng-hau,synthetic,translate the following from English into Hausa.,I'm learning how to cook rice perfectly.,Ina koyon yadda ake dafa shinkafa sosai.


In [117]:
from sklearn.utils import shuffle

# Load the Swahili dataset from a local CSV file
mt_train = pd.read_parquet("hf://datasets/lelapa/MTTrain/data/train-00000-of-00001.parquet")

hau_dataset_pd = mt_train.loc[mt_train['langs'] == 'eng-hau']
#hau_dataset_pd_clean = preprocess_dataframe(hau_dataset_pd)
hau_dataset_pd_clean = data_augment.preprocess_dataframe(hau_dataset_pd)

hau_supplement_df = pd.concat([df_synthetic_hausa, hau_dataset_pd_clean], axis=0, ignore_index=True)
hau_supplement_df = shuffle(hau_supplement_df, random_state=42).reset_index(drop=True)

#hau_dataset_og = Dataset.from_pandas(
#    mt_train[mt_train['langs']=='eng-hau']
#)

swa_dataset = Dataset.from_pandas(
    mt_train[mt_train['langs']=='eng-swa']
)

hau_supplement = Dataset.from_pandas(hau_supplement_df)


# If you need a DatasetDict to mimic the Hugging Face structure
dataset_dict = DatasetDict({
    "swahili": swa_dataset,
    "hausa": hau_supplement
})

# Print to verify
print(swa_dataset)
print(hau_supplement)

AttributeError: module 'src.data_augment' has no attribute 'preprocess_dataframe'

In [87]:
hau_supplement_df

,ID,task,langs,data_source,instruction,inputs,targets
0,ID_d219052d_dev_mt_eng-hau,mmt,eng-hau,wmt22,translate the following from english into hausa.,birds released by this method tend to return t...,wadannan mutane da aka chafke an yi nasarar ch...
1,bootstrap_hausa_convo_33,mmt,eng-hau,synthetic,translate the following from English into Hausa.,Can I borrow your pen?,Zan iya amfani da alƙalanka?
2,bootstrap_hausa_convo_131,mmt,eng-hau,synthetic,translate the following from English into Hausa.,We should leave early to avoid traffic.,Ya kamata mu tafi da wuri domin gujewa cunkoso.
3,bootstrap_hausa_convo_72,mmt,eng-hau,synthetic,translate the following from English into Hausa.,Can I borrow your pen?,Zan iya amfani da alƙalanka?
4,bootstrap_hausa_convo_78,mmt,eng-hau,synthetic,translate the following from English into Hausa.,I'm going to the market. Do you want anything?,Zan je kasuwa. Kana bukatar wani abu?
...,...,...,...,...,...,...,...
472,bootstrap_hausa_convo_106,mmt,eng-hau,synthetic,translate the following from English into Hausa.,We should leave early to avoid traffic.,Ya kamata mu tafi da wuri domin gujewa cunkoso.
473,ID_78f881d7_dev_mt_eng-hau,mmt,eng-hau,wmt22,translate the following from english into hausa.,"for additional information, timelines and hist...","don ƙarin bayani, lokuta da hotunan tarihi da ..."
474,ID_4c8dcfd3_dev_mt_eng-hau,mmt,eng-hau,wmt22,translate the following from english into hausa.,"for a long time, the two old men said",zakaji da yawan samari na ce ma yan
475,ID_53f95d6f_dev_mt_eng-hau,mmt,eng-hau,wmt22,please convert this english content into hausa.,title: how to get along with women,"darasi: yadda mata zaku zauna da uwar miji, a ..."


In [88]:
import csv
from collections import Counter
import numpy as np

def get_char_ngrams(sentence, n):
    sentence = sentence.replace(" ", "")
    return [sentence[i: i + n] for i in range(len(sentence) - n + 1)]

def precision_recall(reference, hypothesis, n):
    ref_ngrams = Counter(get_char_ngrams(reference, n))
    hyp_ngrams = Counter(get_char_ngrams(hypothesis, n))

    common = ref_ngrams & hyp_ngrams
    tp = sum(common.values())

    precision = tp / max(len(hyp_ngrams), 1)
    recall = tp / max(len(ref_ngrams), 1)

    return precision, recall

def f_score(precision, recall, beta=2):
    if precision + recall == 0:
        return 0.0
    return (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall)

def chrF(reference, hypothesis, max_n=6, beta=2):
    precisions, recalls = [], []
    for n in range(1, max_n + 1):
        p, r = precision_recall(reference, hypothesis, n)
        precisions.append(p)
        recalls.append(r)
    avg_p = sum(precisions) / max_n
    avg_r = sum(recalls) / max_n
    return f_score(avg_p, avg_r, beta)


def evaluate_mt_hausa_only(csv_file_path):
    chrfs_scores = []

    with open(csv_file_path, newline="", encoding="utf-8") as csvfile:
        reader = csv.DictReader(csvfile)

        for row in reader:
            # Only evaluate Hausa MT entries
            if "mt" in row["ID"] and "eng-hau" in row["Langs"]:
                ref = row["Targets"]
                pred = row["Response"]
                score = chrF(ref, pred)
                chrfs_scores.append(score)

    avg_chrfs = np.mean(chrfs_scores)
    return round(avg_chrfs, 4), len(chrfs_scores)

In [108]:
import re
import time
# Helper to clean up model outputs
def clean_output(text):
    # Cut off anything before/after the actual translation
    if "Hausa:" in text:
        text = text.split("Hausa:")[-1]

    # Remove prompt leakage
    text = re.sub(r"below is an instruction.*", "", text, flags=re.IGNORECASE)

    # Remove emojis and corrupted unicode
    text = re.sub(r"[^\x00-\x7F]+", "", text)

    # Remove excessive repetition like "ewa, ewa"
    text = re.sub(r"\b(ẽwa[,\.]?\s*){2,}", " ", text, flags=re.IGNORECASE)

    return text.strip()

# Run inference for a specific task
def main(
    model,
    tokenizer,
    BASE_PROMPT,
    task_instruction,
    dataset,
    csv_file_path,
    custom_instruct=False,
    sample_size=4,
    max_new_tokens=100,
    seed=42,
    do_sample=True,
    min_length=None,
    use_cache=True,
    top_p=1.0,
    temperature=0.5,
    top_k=50,
    repetition_penalty=1.2,
    length_penalty=1,
    **kwargs,
):
    actual_model = getattr(model, "model", model)
    actual_model.eval()

    with open(csv_file_path, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(
            [
                "ID",
                "Instruction",
                "Input Text",
                "Response",
                "Log-Likelihood",
                "Targets",
                "Task",
                "Langs",
            ]
        )

        for i, item in enumerate(dataset):
            if i >= sample_size:
                break

            if not custom_instruct:
                instruction = item["instruction"]
            else:
                instruction = task_instruction
            input_text = item["inputs"]
            labels = item["targets"]
            langs = item["langs"]
            task = item.get("task", "xnli")
            identity = item["ID"]

            user_prompt = BASE_PROMPT.format(f"{instruction}\n{input_text}")
            batch = tokenizer(user_prompt, return_tensors="pt")
            batch = {k: v.to(model.device) for k, v in batch.items()}

            start = time.perf_counter()

            with torch.no_grad():
                outputs = model.generate(
                    **batch,
                    max_new_tokens=max_new_tokens,
                    do_sample=do_sample,
                    top_p=top_p,
                    temperature=temperature,
                    min_length=min_length,
                    use_cache=use_cache,
                    top_k=top_k,
                    repetition_penalty=repetition_penalty,
                    length_penalty=length_penalty,
                    **kwargs,
                )

            raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
            output_text = clean_output(raw_output[len(user_prompt):])

            # Optional: log weird outputs
            if re.search(r"[�💖]", output_text):
                print(f"⚠️ Strange output for ID {identity}: {output_text[:100]}")

            if task != "mmt":
                with torch.no_grad():
                    logits = model(**batch).logits
                    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

                    t_labels = (
                        torch.tensor([0, 1, 2])
                        .unsqueeze(0)
                        .unsqueeze(0)
                        .expand(batch["input_ids"].size(0), batch["input_ids"].size(1), -1)
                        .to(model.device)
                    )

                    log_likelihoods_per_class = log_probs.gather(2, t_labels).mean(dim=1)
            else:
                log_likelihoods_per_class = []

            writer.writerow(
                [
                    identity,
                    instruction,
                    input_text,
                    output_text,
                    log_likelihoods_per_class,
                    labels,
                    task,
                    langs,
                ]
            )


In [109]:
# don't change this instruction
mt_instruction = "translate the following from {input_lang} into {output_lang}."
hau_mt_instruction = mt_instruction.format(input_lang="English", output_lang="Hausa")
main(
    model,
    tokenizer,
    BASE_PROMPT,
    sample_size=100,
    task_instruction=hau_mt_instruction,
    dataset=hau_supplement,
    csv_file_path=os.path.join(
        output_path,
        "hau_mt_prediction_dev.csv"
    ),
    custom_instruct=False,
)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

In [111]:
hau_test = pd.read_csv(os.path.join(output_path,"hau_mt_prediction_dev.csv"))

In [112]:
hau_test

,ID,Instruction,Input Text,Response,Log-Likelihood,Targets,Task,Langs
0,ID_d219052d_dev_mt_eng-hau,translate the following from english into hausa.,birds released by this method tend to return t...,edith he was born and died in hausa.,[],wadannan mutane da aka chafke an yi nasarar ch...,mmt,eng-hau
1,bootstrap_hausa_convo_33,translate the following from English into Hausa.,Can I borrow your pen?,ediren kakafi yeem.,[],Zan iya amfani da alƙalanka?,mmt,eng-hau
2,bootstrap_hausa_convo_131,translate the following from English into Hausa.,We should leave early to avoid traffic.,"wa mai karatu ya san cewa, littafin da aka rub...",[],Ya kamata mu tafi da wuri domin gujewa cunkoso.,mmt,eng-hau
3,bootstrap_hausa_convo_72,translate the following from English into Hausa.,Can I borrow your pen?,NaN,[],Zan iya amfani da alƙalanka?,mmt,eng-hau
4,bootstrap_hausa_convo_78,translate the following from English into Hausa.,I'm going to the market. Do you want anything?,ediren kakan efa.,[],Zan je kasuwa. Kana bukatar wani abu?,mmt,eng-hau
...,...,...,...,...,...,...,...,...
95,ID_090a774d_dev_mt_eng-hau,please convert this english content into hausa.,"meanwhile, the social democratic party (sdp) c...",NaN,[],dan takarar jam'iyyar social democratic party ...,mmt,eng-hau
96,ID_4f0f625f_dev_mt_eng-hau,could you convert this english text to hausa?,he also seems to have some resistance to cold,NaN,[],ita kuwa tana samun haka ta dan juyo ta kalles...,mmt,eng-hau
97,ID_034bfc55_dev_mt_eng-hau,translate the following from english into hausa.,why we ought not to wed the pulpit with politi...,"wa, wa, wa, wa, wa, wa,",[],me ya sa ba za mu jefa kuri'a kan karin kudi g...,mmt,eng-hau
98,ID_7c9b5dbf_dev_mt_eng-hau,please convert this english content into hausa.,so that wolves won't eat it.,edar ye ye ye ye ye ye ye ye ye ye ye ye ye ye...,[],"""looters ci shi kadai.",mmt,eng-hau


In [113]:
chrfs_score, n_examples = evaluate_mt_hausa_only(os.path.join(output_path,"hau_mt_prediction_dev.csv"))
print(f"Hausa MT chrF score: {chrfs_score} (based on {n_examples} examples)")


Hausa MT chrF score: 0.1405 (based on 58 examples)


In [ ]:
swa_mt_instruction = mt_instruction.format(input_lang="English", output_lang="Swahili")
model_function.main(
    model,
    tokenizer,
    BASE_PROMPT,
    sample_size=3,
    task_instruction=swa_mt_instruction,
    dataset=swa_dataset,
    csv_file_path=os.path.join(
        output_path,
        "swa_mt_prediction_dev.csv"
    ),
    custom_instruct=False,
)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


XNLI baseline

In [102]:
xnli_train = pd.read_parquet("hf://datasets/lelapa/XNLITrain/data/train-00000-of-00001.parquet")
hau_dataset = Dataset.from_pandas(
    xnli_train[xnli_train['langs']=='hau']
)
swa_dataset = Dataset.from_pandas(
    xnli_train[xnli_train['langs']=='swa']
)

# If you need a DatasetDict to mimic the Hugging Face structure
dataset_dict = DatasetDict({
    "swahili": swa_dataset,
    "hausa": hau_dataset
})

# Print to verify
print(swa_dataset)
print(hau_dataset)

Dataset({
    features: ['ID', 'langs', 'premise', 'inputs', 'instruction', 'targets', '__index_level_0__'],
    num_rows: 200
})
Dataset({
    features: ['ID', 'langs', 'premise', 'inputs', 'instruction', 'targets', '__index_level_0__'],
    num_rows: 200
})


In [103]:
xnli_train

,ID,langs,premise,inputs,instruction,targets
0,ID_648d37ff_dev_afrixnli_hau,hau,"Kuma kawai sai naji Hakane, wannan din ne.","Bayan nace e, ya ƙare.","Is the following question True, False or Neither?",0
1,ID_f96a39cb_dev_afrixnli_hau,hau,"Eh, Eh, na sani, ba zan ma damu sosai ba idan ...",Ba zan damu ba idan ɗaukan nauyin kuɗin da ake...,"Is the following question True, False or Neither?",0
2,ID_4c5d953b_dev_afrixnli_swa,swa,"Alafu, hakuelewa kabisa.","Alas, hakuweza kuelewa wazi kutokana na kizuiz...","Is the following question True, False or Neither?",1
3,ID_99a61e3d_dev_afrixnli_hau,hau,"Kuma kawai sai naji Hakane, wannan din ne.",Nace a'a amma yaci gaba da ja-in-ja.,"Is the following question True, False or Neither?",2
4,ID_a38aa9d8_dev_afrixnli_swa,swa,Lakini siamini chombo chochote cha algorithmic...,Kanuni haziwezi kuamua jinsi ya kutengeneza mk...,"Is the following question True, False or Neither?",1
...,...,...,...,...,...,...
395,ID_085354e1_dev_afrixnli_swa,swa,tahadhari kuhusu jinsi habari za kitaifa zinav...,Maduka ya habari ya kitaifa hufanya maeneo yet...,"Is the following question True, False or Neither?",1
396,ID_586e104a_dev_afrixnli_swa,swa,Uanachama ulijumuisha kati ya wanaume wazima ...,Kurasa zilihusisha wanachama na maafisa wa kaw...,"Is the following question True, False or Neither?",0
397,ID_b871ea53_dev_afrixnli_hau,hau,Ka damu da yadda labarun ƙasa ke shafar unguwa...,Ban damu ba da abinda labarun ƙasa ke nuni cik...,"Is the following question True, False or Neither?",2
398,ID_70aae970_dev_afrixnli_hau,hau,Kuma mani rashin mutunci dan bazan goyi banyan...,Ya zabi ƙin kama hannayen sa saboda anyi musu ...,"Is the following question True, False or Neither?",0


In [ ]:
xnli_instruction = "Is the following question True, False or Neither?"

In [ ]:
# for hausa
model_function.main(
    model,
    tokenizer,
    BASE_PROMPT,
    sample_size=3,
    task_instruction=xnli_instruction,
    dataset=hau_dataset,
    csv_file_path=os.path.join(
        output_path,
        "hau_xnli_prediction_dev.csv"
    ),
    max_new_tokens=15,
    custom_instruct=False,
)

# for swahili
model_function.main(
    model,
    tokenizer,
    BASE_PROMPT,
    sample_size=3,
    task_instruction=xnli_instruction,
    dataset=swa_dataset,
    csv_file_path=os.path.join(
        output_path,
        "swa_xnli_prediction_dev.csv"
    ),
    max_new_tokens=15,
    custom_instruct=False,
)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [27]:
from collections import Counter
def process_likelihood(likelihood_str: str) -> List[float]:
    """
    Process a likelihood string to clean and convert it to a list of floats.
    """
    # Clean the string to remove unwanted characters
    clean_str = (
        likelihood_str.replace("tensor(", "").replace(")", "").strip()
        .replace("[[", "").replace("]]", "").strip()
        .replace(" device='cuda:0'", "").replace(" dtype=torch.float16", "").strip()
        .replace("tensor", "").strip()
    )

    # Remove any empty strings caused by extra commas
    clean_str = clean_str.replace(",,", ",")  # Remove duplicate commas if they exist

    # Convert to a list of floats
    likelihood = [
        float(x) for x in clean_str.split(",") if x.strip()
    ]  # Ensure non-empty strings are converted
    return likelihood

def create_submission(output_path, test_flag: bool):
    """
    Creates submission files based on the provided test_flag.

    Args:
    test_flag (bool): If True, creates a test submission file; otherwise, creates a final submission file.
    """
    if test_flag:
        try:
            df1 = pd.read_csv(os.path.join(
                output_path,
                "hau_sent_prediction_dev.csv")
                 )
            df2 = pd.read_csv(os.path.join(
                output_path,
                "swa_sent_prediction_dev.csv")
            )
            df3 = pd.read_csv(os.path.join(
                output_path,
                "hau_mt_prediction_dev.csv")
                             )
            df4 = pd.read_csv(os.path.join(
                output_path,
                "swa_mt_prediction_dev.csv"))
            df5 = pd.read_csv(os.path.join(
                output_path,
                "hau_xnli_prediction_dev.csv"))
            df6 = pd.read_csv(os.path.join(
                output_path,
                "swa_xnli_prediction_dev.csv"))
        except FileNotFoundError as e:
            print(
                "Seems you have not completed all the tasks, please complete all the tasks before attempting to create your submission file"
            )
            raise e
    else:
        filename = "submission.csv"
        try:
            df1 = pd.read_csv(os.path.join(
                output_path,
                "hau_sent_prediction.csv"))
            df2 = pd.read_csv(os.path.join(
                output_path,
                "swa_sent_prediction.csv"))
            df3 = pd.read_csv(os.path.join(
                output_path,
                "hau_mt_prediction.csv"))
            df4 = pd.read_csv(os.path.join(
                output_path,
                "swa_mt_prediction.csv"))
            df5 = pd.read_csv(os.path.join(
                output_path,
                "hau_xnli_prediction.csv"))
            df6 = pd.read_csv(os.path.join(
                output_path,
                "swa_xnli_prediction.csv"))
        except FileNotFoundError as e:
            print(
                "Seems you have not completed all the tasks, please complete all the tasks before attempting to create your submission file"
            )
            raise e

    # Combine and process data
    resmt = pd.concat([df3, df4], ignore_index=True)
    res_log = pd.concat([df1, df2, df5, df6], ignore_index=True)
    res_log.drop(columns=["Response"], inplace=True)
    res_log.rename(columns={"Log-Likelihood": "Response"}, inplace=True)
    res = pd.concat([res_log, resmt], ignore_index=True)

    def process_row(row):
        if "xnli" in row["ID"] or "sent" in row["ID"]:
            likelihoods = process_likelihood(row["Response"])
            predicted_label = np.argmax(likelihoods)
            return predicted_label
        return row["Response"]  # Default for other cases

    # Update the Response column in-place
    res["Response"] = res.apply(process_row, axis=1)

    if test_flag:
        filename = os.path.join(
                output_path,
                "submission_test.csv")
        # Save the submission file
        submission = res[["ID", "Response", "Targets"]]
        submission.to_csv(filename, index=False)
    else:
        filename = os.path.join(
                output_path,
                "submission.csv")
        # Save the submission file
        submission = res[["ID", "Response"]]
        submission.to_csv(filename, index=False)
    return submission

def evaluate_zindi_by_language(df):
    from collections import defaultdict, Counter
    import numpy as np

    results = defaultdict(lambda: defaultdict(list))  # results[task][lang] = [(true, pred), ...]
    chrfs_scores = defaultdict(list)

    for _, row in df.iterrows():
        ID = row["ID"].lower()

        # --- Task detection ---
        if "sentiment" in ID or "sent" in ID:
            task = "sent"
        elif "xnli" in ID or "afrixnli" in ID:
            task = "xnli"
        elif "mt" in ID:
            task = "mt"
        else:
            continue  # skip unknown tasks

        # --- Language detection ---
        if any(x in ID for x in ["swahili", "swa", "eng-swa"]):
            lang = "swahili"
        elif any(x in ID for x in ["hausa", "hau", "eng-hau"]):
            lang = "hausa"
        else:
            lang = "unknown"

        # --- Scoring logic ---
        if task == "sent":
            labels = {
                "swahili": ["Chanya", "Wastani", "Hasi"],
                "hausa": ["Kyakkyawa", "Tsaka-tsaki", "Korau"]
            }.get(lang, [])
            if not labels:
                continue
            label_to_id = {label.lower(): i for i, label in enumerate(labels)}
            true = label_to_id.get(row["Targets"].strip().lower(), -1)
            pred = int(row["Response"])
            if true != -1:
                results[task][lang].append((true, pred))

        elif task == "xnli":
            try:
                true = int(row["Targets"])
                pred = int(row["Response"])
                results[task][lang].append((true, pred))
            except:
                continue  # skip rows with invalid labels

        elif task == "mt":
            ref = row["Targets"]
            hyp = row["Response"]
            chrfs = chrF(reference=ref, hypothesis=hyp)
            chrfs_scores[lang].append(chrfs)

    # --- Aggregation ---
    report = {}
    for task in results:
        for lang in results[task]:
            y_true, y_pred = zip(*results[task][lang])
            f1 = calculate_f1(np.array(y_true), np.array(y_pred), num_classes=3)
            report[f"{task}_{lang}"] = round(f1, 4)

    for lang in chrfs_scores:
        avg_chrfs = np.mean(chrfs_scores[lang])
        report[f"mt_{lang}"] = round(avg_chrfs, 4)

    # --- Zindi score ---
    zindi_score = np.mean(list(report.values()))
    report["zindi_score"] = round(zindi_score, 4)

    return report


# From scratch implementation of chrf
def get_char_ngrams(sentence, n):
    """Generate character n-grams from a sentence."""
    sentence = sentence.replace(" ", "")  # Remove spaces for chrF
    return [sentence[i : i + n] for i in range(len(sentence) - n + 1)]


def precision_recall(reference, hypothesis, n):
    """Calculate precision and recall for character n-grams."""
    ref_ngrams = get_char_ngrams(reference, n)
    hyp_ngrams = get_char_ngrams(hypothesis, n)

    ref_count = Counter(ref_ngrams)
    hyp_count = Counter(hyp_ngrams)

    common_ngrams = ref_count & hyp_count
    true_positives = sum(common_ngrams.values())

    precision = true_positives / max(len(hyp_ngrams), 1)
    recall = true_positives / max(len(ref_ngrams), 1)

    return precision, recall


def f_score(precision, recall, beta=1):
    """Calculate the F1 score."""
    if precision + recall == 0:
        return 0.0
    return (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall)


def chrF(reference, hypothesis, max_n=6, beta=2):
    """Calculate the chrF score from scratch."""
    precisions = []
    recalls = []

    for n in range(1, max_n + 1):
        precision, recall = precision_recall(reference, hypothesis, n)
        precisions.append(precision)
        recalls.append(recall)

    avg_precision = sum(precisions) / max_n
    avg_recall = sum(recalls) / max_n

    return f_score(avg_precision, avg_recall, beta)


# From scratch implementation f1score 3 class
def calculate_f1(true_labels, pred_labels, num_classes):
    f1_scores = []

    for i in range(num_classes):
        TP = np.sum((true_labels == i) & (pred_labels == i))  # True Positives
        FP = np.sum((true_labels != i) & (pred_labels == i))  # False Positives
        FN = np.sum((true_labels == i) & (pred_labels != i))  # False Negatives

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0

        # Calculate F1 score
        f1 = (
            2 * (precision * recall) / (precision + recall)
            if (precision + recall) > 0
            else 0
        )
        f1_scores.append(f1)

    macro_f1 = np.mean(f1_scores)

    return macro_f1


In [28]:
score_test = create_submission(output_path=output_path, test_flag=True)

Seems you have not completed all the tasks, please complete all the tasks before attempting to create your submission file


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/InkubaLM-Challenge/Output/baseline_performance/datasets/swa_mt_prediction_dev.csv'

In [29]:
score_test

NameError: name 'score_test' is not defined

In [ ]:
print(score_test["ID"].unique().tolist())

['ID_6aba33a1_sentiment_ dev_hausa', 'ID_ce64d307_sentiment_ dev_hausa', 'ID_2efc9515_sentiment_ dev_hausa', 'ID_dfb02831_sentiment_ dev_swahili', 'ID_ad1d9888_sentiment_ dev_swahili', 'ID_34eed91d_sentiment_ dev_swahili', 'ID_648d37ff_dev_afrixnli_hau', 'ID_f96a39cb_dev_afrixnli_hau', 'ID_99a61e3d_dev_afrixnli_hau', 'ID_4c5d953b_dev_afrixnli_swa', 'ID_a38aa9d8_dev_afrixnli_swa', 'ID_643fa526_dev_afrixnli_swa', 'ID_5e3d2251_dev_mt_eng-hau', 'ID_fd263d98_dev_mt_eng-hau', 'ID_96a1f4d4_dev_mt_eng-hau', 'ID_4185b417_dev_mt_eng-swa', 'ID_a694c64e_dev_mt_eng-swa', 'ID_13e437e9_dev_mt_eng-swa']


In [30]:
score_report = evaluate_zindi_by_language(score_test)
score_report

NameError: name 'score_test' is not defined

In [ ]:
# 1. How many rows of xnli_hausa?
df = score_test
df[(df["ID"].str.contains("xnli", case=False) | df["ID"].str.contains("afrixnli", case=False)) &
   (df["ID"].str.contains("hau", case=False))]


,ID,Response,Targets
6,ID_648d37ff_dev_afrixnli_hau,1,0
7,ID_f96a39cb_dev_afrixnli_hau,1,0
8,ID_99a61e3d_dev_afrixnli_hau,1,2
